In [ ]:
Analysis of aging-related differential gut microbial changes in public gut metagenomic datasets

Public gut metagenomic datasets from Gianmarco Piccinno et al. were utilized to investigate aging-related differential gut microbial changes. Individuals were filtered from the datasets based on the following criteria: (1) comprehensive metadata (e.g., age, sex, BMI) were available for each individual; (2) each individual was classified as exclusively healthy. Ultimately, we obtained a matrix comprising 1519 individuals from 15 studies, spanning 10 countries, including Austria, India, China, Italy, the United States, Germany, Japan, France, Czechia, and Turkey.

In [ ]:
rm(list=ls())
library(tidyverse)
library(ggplot2)
library(ggsci)
library(dplyr)
library(patchwork)
library(microbiome)
library(vegan)
library(ecodist)

list_files = list.files(path = "./15069069/metadata/")
res=c()
for (i in list_files){
  feature <- read.csv(paste0("./15069069/metadata/",i),sep="\t")
  print("**************************")
  print(i)
  print(table(feature$Control))
  
  feature <-   feature %>%
    filter(Control == "Study Control")

  print(table(feature$Control))
  print(fivenum(feature$Age))
  if(dim(feature)[1] > 0){
    res<-rbind(res,feature)
  }
}


df_input_metadata<-res%>%
  # filter(Treat !="Used")%>%
  # filter(Group =="Control")%>%
  filter(Name %in% colnames(df_input_data))%>% 
  filter(!is.na(Age))%>% 
  filter(!is.na(BMI))


To identify the enterotype in this cohort, we used PAM analysis based on the relative abundance of microbial genera and used the Calinski-Harabasz index to select the optimal number of clusters. 

In [ ]:
rm(list=ls())
library(tidyverse)
library(ggplot2)
library(ggsci)
library(dplyr)
library(patchwork)
library(ggpubr)
library(cluster)
library(clusterSim)
library(ade4)

setwd("E:/QZ/外部队列/ColorectalCancer/02_Enterotype")
getwd()

data <- read.csv("E:/QZ/外部队列/ColorectalCancer/merge_table_species.csv",row.names = 1,check.names = F)

data[1:5, 1:5]

metadata <- read.csv("../ColorectalCancer_metadata.csv")%>%
  # filter(Treat !="Used")%>%
  # filter(Group =="Control")%>%
  filter(Name %in% colnames(data))
  

fivenum(metadata$Age)

colSums(data)
data=data[,intersect(colnames(data),metadata$Name)]
# noise.removal <- function(dataframe, percent=0.001, top=NULL){
#   dataframe->Matrix
#   bigones <- rowSums(Matrix)*100/(sum(rowSums(Matrix))) > percent 
#   Matrix_1 <- Matrix[bigones,]
#   print(percent)
#   return(Matrix_1)
# }
# 
# data=noise.removal(data, percent=0.001)

dist.JSD <- function(inMatrix, pseudocount=0.000001, ...) {
  KLD <- function(x,y) sum(x *log(x/y))
  JSD<- function(x,y) sqrt(0.5 * KLD(x, (x+y)/2) + 0.5 * KLD(y, (x+y)/2))
  matrixColSize <- length(colnames(inMatrix))
  matrixRowSize <- length(rownames(inMatrix))
  colnames <- colnames(inMatrix)
  resultsMatrix <- matrix(0, matrixColSize, matrixColSize)
  
  inMatrix = apply(inMatrix,1:2,function(x) ifelse (x==0,pseudocount,x))
  
  for(i in 1:matrixColSize) {
    for(j in 1:matrixColSize) { 
      resultsMatrix[i,j]=JSD(as.vector(inMatrix[,i]),
                             as.vector(inMatrix[,j]))
    }
  }
  colnames -> colnames(resultsMatrix) -> rownames(resultsMatrix)
  as.dist(resultsMatrix)->resultsMatrix
  attr(resultsMatrix, "method") <- "dist"
  return(resultsMatrix) 
}

##PAM??????Ʒ
pam.clustering=function(x,k) { # x is a distance matrix and k the number of clusters
  require(cluster)
  cluster = pam(as.dist(x), k, diss=TRUE)$clustering
  # cluster = as.vector(pam(as.dist(x), k, diss=TRUE)$clustering)
  return(cluster)
}
data.dist=dist.JSD(data)

data.cluster=pam.clustering(data.dist, 
                            k=2) # k=3 Ϊ??

nclusters = index.G1(t(data), data.cluster, d = data.dist, centrotypes = "medoids") # ?鿴CHָ??
nclusters = NULL

# for(k in 1:20)
for(k in 1:10)
{ 
  if(k==1)
  {
    nclusters[k] = NA 
  }
  else
  {
    data.cluster_temp = pam.clustering(data.dist, k)
    nclusters[k] = index.G1(t(data), data.cluster_temp,  d = data.dist, centrotypes = "medoids")
  }
}
pdf("CH_index.pdf",height=4,width=5)
nclusters[1] = 0
plot(nclusters, type="b", xlab="k clusters", ylab="CH index",lwd = 2)
# plot(nclusters, type="h", xlab="k clusters", ylab="CH index", main="Optimal number of clusters") # ?鿴K??CHֵ?ù?ϵ
dev.off()


# k_best = which(nclusters == 3, arr.ind = TRUE)
k_best = which(nclusters == max(nclusters), arr.ind = TRUE)
k_best
k_best=2
data.cluster=pam.clustering(data.dist, k = 2)

mean(silhouette(data.cluster, data.dist)[, 3])

obs.pca=dudi.pca(data.frame(t(data)), scannf=F, nf=2)
obs.bet=bca(obs.pca, fac=as.factor(as.vector(data.cluster)), scannf=F, nf=2) 
head(obs.bet$tab)
write.table(t(obs.bet$tab),"bca_table_drivers.txt",sep = '\t',quote=F)

obs.pcoa=dudi.pco(data.dist, scannf=F, nf=2)

pdf("enterotype_label.pdf")
s.label(obs.pcoa$li,grid=F,cl = 0.5, cp = 3)
s.class(obs.pcoa$li, fac=as.factor(as.vector(data.cluster)), 
        # label = c("E1","E2","E3"),
        cl = 2,xlim = c(-0.5,0.5),ylim=c(-0.5,0.5),
        grid=F, 
        col=c("#f5989a",   "#9fbcdb"),
        # "#FF9896","#DBDB8D","#9EDAE5"
        add.plot = TRUE)
dev.off()

pdf("enterotype.pdf")
s.class(obs.pcoa$li, fac=as.factor(as.vector(data.cluster)),
        # label = c("E1","E2","E3"),
        cl = 2,xlim = c(-0.5,0.5),ylim=c(-0.5,0.5),
        col=c("#f5989a",   "#9fbcdb"),
        grid=F)
dev.off()
write.csv(data.cluster,"data.cluster.csv",quote=F)
write.csv(obs.pcoa$li,"obs.pcoa_li.csv",quote=F)

In [ ]:
Then, we used MaAsLin2 (version 1.16.0) in R, which finds associations between enterotype-independent microbial features and age, and MaAsLin2 uses a transformed generalized linear model to associate each feature iteratively with covariates of interest.
Feature ≈ age + enterotype + sex + BMI + study
False discovery rate (FDR, q value) less than 0.1 was set as the threshold for statistical significance.

In [ ]:
rm(list=ls())
library(tidyverse)
library(ggplot2)
library(ggsci)
library(dplyr)
library(patchwork)
library(Maaslin2)

setwd("E:/QZ/外部队列/ColorectalCancer/07_Taxonomy/MaAsLin2/Species")
getwd()

df_input_data <- read.csv("E:/QZ/外部队列/ColorectalCancer/merge_table_species.csv",row.names = 1,check.names = F)

df_input_data[1:5, 1:5]

df_input_metadata<-read.csv("E:/QZ/外部队列/ColorectalCancer/ColorectalCancer_metadata.csv",row.names = 1)%>%
  # filter(Treat !="Used")%>%
  # filter(Group =="Control")%>%
  filter(Name %in% colnames(df_input_data))%>% 
  filter(!is.na(Age))%>% 
  filter(!is.na(BMI))

Enterotype_cluster = read.csv(
  "E:/QZ/外部队列/ColorectalCancer/02_Enterotype/data.cluster.csv")%>%
  mutate(Enterotype = ifelse(x == 1, "ETP","ETB"))


df_input_metadata<-merge(Enterotype_cluster,df_input_metadata,
            by.x = "X", by.y = "Name")

rownames(df_input_metadata)<-df_input_metadata$X
# %>%filter(Sex=="Male")
table(df_input_metadata$Study.name)
# table(df_input_metadata$Sex)
# table(df_input_metadata$Batch)
intersect(rownames(df_input_metadata),colnames(df_input_data))
df_input_data<-df_input_data[,intersect(rownames(df_input_metadata),colnames(df_input_data))]
# head(df_input_metadata)
# colnames(df_input_metadata)
# table(df_input_metadata$Sex)
df_input_metadata$Study.name<-factor(df_input_metadata$Study.name)
fit_data = Maaslin2(input_data     = df_input_data,
                    input_metadata = df_input_metadata,
                    min_prevalence = 0.05,
                    # normalization  = "NONE",
                    # standardize = FALSE,  
                    output         = "Result", plot_scatter = F,
                    # random_effects = c("Country"),
                    # cores = 4 ,
                    fixed_effects  = c("Age",
                                       "BMI",
                                       "Sex",
                                       "Enterotype",
                                       "Study.name"
                                       ))
